In [1]:
import warnings
from causalnex.structure import StructureModel
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [2]:
df_se = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_2007_2015_bn.csv")

In [3]:
df_se = df_se.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_se

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,DI,GE,CPI,RQ,VAA,EUPC
0,9.42,5.65,6.03,2.25,22.0,0.74,25.02,24.077797,24.205811,2.462276,...,76.5,0.07,1920.24,-2007.0,9.73,2.02,93.0,1.57,1.51,3.554298e+06
1,8.05,5.63,6.02,2.12,23.0,0.78,24.15,23.960719,24.785451,-1.691966,...,76.7,0.06,1074.44,-1243.0,9.88,1.95,93.0,1.64,1.52,5.290971e+06
2,2.06,5.69,6.04,2.18,23.0,0.85,22.33,20.459961,26.132116,-5.067760,...,75.1,0.01,1618.18,-3953.0,9.73,2.04,92.0,1.65,1.55,1.950399e+07
3,0.13,5.71,6.20,2.10,23.0,0.85,23.86,22.421542,25.187785,4.853017,...,74.4,0.03,1729.81,5519.0,9.50,1.96,92.0,1.66,1.54,1.868423e+07
4,1.22,5.66,6.25,1.99,23.0,0.76,23.73,23.236959,25.092481,2.387802,...,76.0,0.06,1966.61,-3806.0,9.50,1.93,93.0,1.89,1.61,3.370849e+07
5,0.78,5.58,5.97,2.06,23.0,0.74,23.11,21.993432,25.878585,-1.148445,...,76.4,0.03,1723.00,-2826.0,9.73,1.91,88.0,1.90,1.69,1.155998e+07
6,0.25,5.52,5.73,2.03,24.0,0.85,21.98,21.899036,26.318281,0.284348,...,76.7,0.06,1949.87,-1789.0,9.73,1.79,89.0,1.90,1.66,2.608015e+07
7,-1.05,5.50,5.67,1.97,24.0,0.99,21.61,22.864139,26.275487,1.285784,...,77.2,0.04,650.56,-1307.0,9.73,1.68,87.0,1.81,1.61,3.553847e+07
8,1.95,5.36,5.60,2.04,24.0,1.06,22.28,23.788437,25.892770,3.311868,...,77.9,0.05,5207.32,-20.0,9.45,1.68,89.0,1.80,1.57,8.000893e+06


In [5]:
se = pd.DataFrame()
se = df_se.copy()
se['VABI_3'] = se['VABI'].shift(3)
se['PDPS_3'] = se['PDPS'].shift(3)
se['EUPC_2'] = se['EUPC'].shift(2)
se['PDPS_2'] = se['PDPS'].shift(2)
se['GE_1'] = se['GE'].shift(1)
se['EPFRG_1'] = se['EPFRG'].shift(1)

In [6]:
se = se.dropna()

In [7]:
sm = StructureModel()

In [8]:
sm.add_edges_from([
    ('EUPC_2', 'EPFRG'),
])

In [9]:
sm.edges

OutEdgeView([('EUPC_2', 'EPFRG')])

In [10]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_se_2007_2015.html")

Graphs/fully_connected_se_2007_2015.html


In [11]:
bn = BayesianNetwork(sm)

In [12]:
discretised_se = pd.DataFrame(index=se.index)

for col in se.columns:
    no_unique = se[col].nunique()
    
    if no_unique <= 1:
        discretised_se[col] = 0
    else:
        try:
            discretised_se[col] = pd.qcut(
                se[col], 
                q=min(3, no_unique),
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_se[col] = se[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_se)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  CPI  RQ  VAA  \
3    0    2    2     2     0    1     2     1     0      2  ...    2   0    0   
4    2    2    2     0     0    0     2     2     0      1  ...    2   1    1   
5    1    1    1     2     0    0     1     0     1      0  ...    0   2    2   
6    1    1    1     1     1    1     0     0     2      0  ...    1   2    2   
7    0    0    0     0     1    2     0     1     2      1  ...    0   1    1   
8    2    0    0     1     1    2     1     2     1      2  ...    1   0    0   

   EUPC  VABI_3  PDPS_3  EUPC_2  PDPS_2  GE_1  EPFRG_1  
3     1       2       0       0       0     2        0  
4     2       2       0       1       0     2        1  
5     0       0       0       1       0     1        2  
6     1       1       0       2       0     1        1  
7     2       1       0       0       0     0        2  
8     0       0       0       2       0     0        0  

[6 rows x 33 c

In [13]:
discretised_se = discretised_se.reset_index(drop=True)
bn.fit_node_states(discretised_se)
baseline_auc = utils.get_avg_auc_all_info(discretised_se, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 2.9852333068847656 seconds
Processing fold 1 using 7 cores takes 2.92189621925354 seconds
Processing fold 2 using 7 cores takes 2.847759246826172 seconds
Processing fold 3 using 7 cores takes 2.8857250213623047 seconds
Processing fold 4 using 7 cores takes 2.874627113342285 seconds
Baseline AUC: 0.15


In [14]:
edges_to_add = [('LV', 'EUPC_2'), ('LV', 'EPFRG')]
edges_to_remove = [('EUPC_2', 'EPFRG')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [15]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_se_2005_2015.html")

Graphs/node_added_se_2005_2015.html


In [16]:
discretised_se['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_se, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_2' and 'EPFRG': {proposed_auc}")

Processing fold 0 using 7 cores takes 3.0445570945739746 seconds
Processing fold 1 using 7 cores takes 2.9919722080230713 seconds
Processing fold 2 using 7 cores takes 3.0915918350219727 seconds
Processing fold 3 using 7 cores takes 2.9680418968200684 seconds
Processing fold 4 using 7 cores takes 3.035297155380249 seconds
AUC from adding LV between 'EUPC_2' and 'EPFRG': 0.05
